<h1 style="text-align:center">Recurrent Neural Network</h1>                                                       


## Part 1: Generate text


Recurrent layers are used when the input data is sequential and the prediction of the next element in the sequence heavily depends on the prediction made on the previous elements. Typically, the cases where recurrent layers are most commonly used are:<br>

* A sequence of words arranged sequentially, that is, a text.
* A sequence of frequencies, that is, a sound or a time series.
* A sequence of images, that is, a video.
  
The functioning of a recurrent layer and the intuition behind its creation are very simple.

Suppose we want to develop an algorithm that auto-completes a word a user was typing. A first approach to this problem would be to predict the next character the user was going to type based on the characters they have already written.

For example, if the user wanted to write the word "Hello" and only wrote "Hel", the algorithm should predict the character "l".

In this part of the lab, we are going to set up a recurrent network-based algorithm to address the problem stated in the introduction: **predicting the next character based on the previous characters**.<br>
Training the model will only require a text formatted in a usable way as the dataset.

* Open and read the file shakespeare.txt with 'utf-8' encoding. Store the file in the variable text.<br>
* Display the first 250 characters.

In [4]:
with open("shakespeare.txt", encoding="utf-8") as file:
    text = file.read()

print(text[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



* Store all the unique characters from the text in the variable vocab.Sort the result using **sorted**.<br>
* Display the number of unique characters.<br>
  
The **`set`** command allows you to return the unique elements of a list or a string.

In [5]:
vocab = sorted(set(text))

print(len(vocab))

65


### Convert to the index
As in traditional text mining tasks, the algorithms only process numbers and not textual data. It is therefore necessary to convert each character of our corpus into a number (here, indexes).

* Create a **char2idx** dictionary, in which you will store the character as the key and its index as the value:



In [6]:
char2idx = {char: idx for idx, char in enumerate(vocab)}
char2idx

{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}

Run the following cell to display the first elements of the **char2idx** dictionary.

In [7]:
import numpy as np
print('{')
for char, _ in zip(char2idx, range(10)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')


{
  '\n':   0,
  ' ' :   1,
  '!' :   2,
  '$' :   3,
  '&' :   4,
  "'" :   5,
  ',' :   6,
  '-' :   7,
  '.' :   8,
  '3' :   9,
  ...
}


* Convert all the characters in the text into indices in the list **text_as_int**.

In [8]:
text_as_int = [char2idx[char] for char in text]
text_as_int[:10]

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47]

### Generating the Dataset

* Import **TensorFlow**.<br>
* Create the variable **seq_length** = 100 representing the number of characters per sequence.<br>
* Define a dataset **char_dataset** using the **from_tensor_slices** function from the **text_as_int** dataset.

In [2]:

import tensorflow as tf
seq_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
char_dataset

NameError: name 'text_as_int' is not defined

Run the following cell to display the first 5 elements of **char_dataset** as characters.

In [10]:
idx2char = np.array(vocab)

for i in char_dataset.take(5):
     print(idx2char[i.numpy()])

F
i
r
s
t


We will now apply a series of transformations to shape our dataset.

![image.png](attachment:48850814-90eb-43c1-b5bc-de15c9c14bca.png)

1. The first batch allows the text to be split into parts of **seq_length + 1** characters.
2. The **map** transformation then allows us to separate our sub-texts into input data and target data (shifting the input and output by one character).

Example: For sub-text ['H','o','w',' ','a','r','e',' ','y','o','u'] (size 11) :

features = ['H','o','w',' ','a','r','e',' ','y','o'] (size 10)

targets = ['o','w',' ','a','r','e',' ','y','o', 'u'] (size 10)

3. **Batch** and **shuffle** transformations finally allow you to shape the data into batches and shuffle them.

* Use the batch method applied to **char_dataset**, split the dataset into batches of **length seq_length + 1**. Specify the argument **drop_remainder = True** in the method, indicating that the last batch should be discarded if it is incomplete. Store the result under the name **sequences**.

In [11]:
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)


Run the following cell to display the first 2 elements of sequences as characters.

In [12]:
for item in sequences.take(2):
    print(repr(''.join(idx2char[item.numpy()])))

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
'are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you k'


* Use the **map** method of **char_dataset**, separate the input data and the target data. Store the result under the name **dataset**.

In [13]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

Run the following cell to display the input and target of the first dataset element as characters.

In [14]:

for input_example, target_example in  dataset.take(1):
    print ('Input data: ', repr(''.join(idx2char[input_example.numpy()])))
    print ('Target data:', repr(''.join(idx2char[target_example.numpy()])))


Input data:  'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
Target data: 'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


* Apply the shuffle method to the **dataset** specifying a **buffer_size** of 10000 to shuffle the dataset.<br>
* Apply the batch method to the dataset specifying a **batch_size** of 64 to split the dataset into batches. Specify the argument **drop_remainder = True** in the method (do not take the last incomplete batch).

In [15]:
dataset = dataset.shuffle(10000).batch(64, drop_remainder=True)


## Model: RNN

![image.png](attachment:5f8bfef6-8189-4714-be5d-ca2bc8045245.png)

### Embedding layer
The input of our model is fed by a sequence of character indices. As with categorical variables, it is necessary to transform them into a vector form **(get_dummies, one hot, embedding, etc.)**. Here, we will represent the indices in vector form using the Embedding layer (this layer can be seen as a **one hot + dense layer)**.

Example:

<span style="color:gray">tf.keras.layers.Embedding(input_dim=vocab_size, # maximum index + 1 (vocabulary size)<br>
output_dim=embedding_dim) # size of the output vector

### Recurrent network

The **RNN** constructor from the **layers** submodule allows you to instantiate an **RNN** layer. In the constructor, we will specify the following arguments:

* **cell**: the type of RNN cell (**SimpleRNNCell**, **LSTMCell**, and **GRUCell**)
* **return_sequences**: Indicates whether the layer returns the output sequence h1,...,hn or just the last element hn (model: many-to-one or many-to-many).
* **return_state**: Returns the last state in addition to the output.
* **stateful**: If True, the last state (hidden state) **hn** of each sample in a batch will be used as the initial state **h0** for the next batch.

The **SimpleRNNCell**, **LSTMCell**, and **GRUCell** functions from the **layers** submodule define the type of cell used in the RNN. They take the following arguments:

* **units**: Dimension of the hidden state **h**.
* **activation**: Activation function for the cell output.
* **use_bias**: Whether to use biases.
* **kernel_initializer**: Method for initializing the weights.
* **bias_initializer**: Method for initializing the bias.

![image.png](attachment:5a051e8f-a8e9-4d6e-9a49-ebc4aadc0a72.png)

Example:

<span style="color:gray">tf.keras.layers.RNN(    

<span style="color:gray">                          cell=LSTMCell(128),           # RNN computation cell

<span style="color:gray">                          return_sequences=True)        # Returns a sequence

* Set the variable vocab_size as the number of elements in our dictionary.<br>
* Define a function **build_model** with **batch_size** as an argument:<br>
1. Instantiate a sequential model called **model** using the **Sequential** constructor from **tensorflow.keras**.<br>
2. Define an Embedding layer specifying an **input size** of **vocab_size** and an output size of 256.<br>
3. Furthermore, since we will be using **stateful = True** in the RNN later, it is necessary to specify the argument **batch_input_shape = [batch_size, None]**.<br>
4. Add an RNN layer to the model with a **GRUCell** of dimension 512 that returns a sequence and reuses the last state as the initial state ℎ0.<br>
5. Add a Dense layer to the model with **vocab_size** neurons and a **'softmax' activation function**.<br>
6. Return the model.

In [16]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, RNN, GRUCell

vocab_size = len(vocab)

def build_model(batch_size):

    model = tf.keras.Sequential()

    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=256
    ))


    model.add(RNN(
        GRUCell(512),
        return_sequences=True,
        stateful=True
    ))

    model.add(Dense(
        vocab_size,
        activation='softmax'
    ))

    return model

* Define a model under the name **model** using the **build_model** function with a **batch_size** of 64.<br>
* Compile the model using its **compile** method, specifying the **Adam** optimizer with a learning rate of **1e-3** and the **sparse_categorical_crossentropy** loss function. Display the model summary.

In [17]:
model = build_model(batch_size=64)
model.summary()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy'
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_1 (RNN)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

* Train the model on the **dataset** data for 30 epochs using the fit method.

In [18]:

history = model.fit(
    dataset,
    epochs=30
)

Epoch 1/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - loss: 2.9595
Epoch 2/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 1.9817
Epoch 3/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 1.7353
Epoch 4/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 1.5939
Epoch 5/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - loss: 1.5105
Epoch 6/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - loss: 1.4568
Epoch 7/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.4164
Epoch 8/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.3861
Epoch 9/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.3567
Epoch 10/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 12s 58ms/step - loss: 1.3390
Epoch 11/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - loss: 1.3220
Epoch 12/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.3060
Epoch 13/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.2888
Epoch 14/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.2731
Epoch 15/30
172/172 ━━━━━━━━━━━━━━━

Here, the batch size is fixed, so it is not possible to generate texts with batches other than 64. To work around this limitation, that is, to process a single text, you can use the following trick:

1. Save the weights of our model using the **save_weights** method.
2. Reload a new model with a **batch_size** of 1.
3. Load the weights of the old model using the **load_weights** method.<br>
   
* Save the model's weights using the **save_weights** method, specifying the path **"model_rnn.h5"**.
* Create a new model named model using the **build_model** function, specifying a **batch_size** of 1.
* Load the weights of our trained model using the **load_weights** method, specifying the path "**model_rnn.h5**".

In [21]:
import numpy as np

model.save_weights("temp_weights.weights.h5")

model_single = build_model(batch_size=1)

dummy_input = np.zeros((1, 1), dtype=np.int32)
model_single(dummy_input)

model_single.load_weights("temp_weights.weights.h5")

model_single.save_weights("model_rnn.weights.h5")

Run the following cell to generate a text starting with "**ROMEO:**"

In [23]:
def generate_text(model, start_string, num_generate=500):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []
    for layer in model.layers:
        if hasattr(layer, "reset_states"):
            layer.reset_states()

    for i in range(num_generate):
        prediction = model(input_eval)
        prediction = prediction[:, -1, :]
        index = tf.argmax(prediction, axis=-1).numpy()[0]
        input_eval = tf.expand_dims([index], 0)
        text_generated.append(idx2char[index])
    return start_string + ''.join(text_generated)

print(generate_text(model, start_string="ROMEO:"))


ROMEO:
I think there be the man of the fardel of my son,
And therefore I'll swear to thee, the deed that thou hast made thee to the good of the world.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done


## Part 2: RNN Embedding - Classification

Recurrent layers are used when the input data is sequential. In other words, this layer captures sequential relationships in the input data. This property is particularly useful with textual data. Indeed, the order of words can change the meaning of a sentence or even the sense of the words.

**Example**: two sentences with the same words, but different meanings:

"Woman dances well, only for a little while."<br>
"While for a little woman well, only dances."

### DataSet
The goal of this lab is to add the sequential dimension to a sentiment analysis problem. The dataset contains 50,000 movie reviews from the IMDB site labeled as positive/negative.
* Load the file '**MovieReview.csv**' as **df**.<br>
* Replace the values 'Negative' and 'Positive' in the "sentiment" column with 0 and 1.<br>
* Display the first 5 rows of the dataframe.

In [25]:
import pandas as pd
df = pd.read_csv("MovieReview.csv")

df["sentiment"] = df["sentiment"].replace({"Negative": 0, "Positive": 1})

print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


Run the following cell to install the needed tools.

In [26]:
!pip install nltk
!pip install swifter
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=a1c0d7d0eaf13ad151bf19815180cec141884f709828e7f35ac4878cbec3ec74
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter


Run the following cell to clean the data and remove the stopwords.

In [27]:
import re
import unicodedata
import pandas as pd

try:
    import swifter
    use_swifter = True
except ModuleNotFoundError:
    use_swifter = False
    print("Swifter not installed. Using pandas apply instead.")

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, PunktSentenceTokenizer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

stop_words = set(stopwords.words('english'))


def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'\s+', ' ', w)
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    w = re.sub(r'\b\w{0,2}\b', '', w)
    try:
        mots = word_tokenize(w.strip())
    except LookupError:
        tokenizer = PunktSentenceTokenizer()
        mots = []
        for sentence in tokenizer.tokenize(w.strip()):
            mots.extend(sentence.split())
    mots = [mot for mot in mots if mot not in stop_words]

    return ' '.join(mots).strip()

if use_swifter:
    df.review = df.review.swifter.apply(preprocess_sentence)
else:
    df.review = df.review.apply(preprocess_sentence)

df.head()


Pandas Apply:   0%|          | 0/50000 [00:00<?, ?it/s]

,review,sentiment
0,one reviewers mentioned watching episode hooke...,positive
1,wonderful little production . filming techniqu...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake thinks zombie...,negative
4,petter mattei love time money visually stunnin...,positive


* Separate the dataset **df.review** and **df.sentiment** into a training set **X_text_train**, **y_train**, and a test set **X_text_test**, **y_test**<br>
We will choose an 80% ratio for the training data and a random seed of 1234.

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split

X_text = df['review']
y = df['sentiment']

X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.8,
    random_state=1234
)

print("Taille X_train :", len(X_text_train))
print("Taille X_test :", len(X_text_test))

Taille X_train : 10000
Taille X_test : 40000


As in traditional text mining tasks, the algorithms only process numbers and not textual data. Here, character-based approaches are not suitable since the meaning of a word is not really determined by its structure (the words "hat" and "hate" are unrelated even though there is only a one-character difference).

For these reasons, we will work at the word level.

### Tokenizer
The Tokenizer class from **tensorflow.keras.preprocessing.text** allows you to vectorize a text corpus by converting each text into a sequence of integers (each integer being the index of a token in a dictionary).

The **num_words** argument limits the size of the dictionary to this amount. More information here.https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer

<span style="color:gray">tf.keras.preprocessing.text.Tokenizer(<br>
   num_words=None, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n', lower=True,<br>
   split=' ', char_level=False, oov_token=None, document_count=0, **kwargs<br>
)

The **fit_on_texts** method of this object allows our tokenizer to update the word dictionary based on a list of texts.

<span style="color:gray">fit_on_texts(<br>
   texts<br>
)</span><br>

* Define a **tokenizer** object using the **Tokenizer** constructor from **tensorflow.keras.preprocessing.text**, specifying a dictionary word limit of 10,000.<br>
* Update the dictionary of our **tokenizer** from **X_text_train** using the **fit_on_texts method**.

In [29]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=10000,
    lower=True,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_text_train)

print("Nombre de mots dans le dictionnaire :", len(tokenizer.word_index))

Nombre de mots dans le dictionnaire : 50613


* Store the dictionary mapping words to indices in the variable **word2idx** using the tokenizer's **word_index** attribute.<br>
* Store the dictionary mapping indices to words in the variable **idx2word** using the tokenizer's **index_word** attribute.<br>
* Store the size of the dictionary in the variable **vocab_size** using the tokenizer's **num_words** attribute.

In [30]:
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = tokenizer.num_words
print("Exemple word2idx :", list(word2idx.items())[:5])
print("Exemple idx2word :", list(idx2word.items())[:5])
print("Taille du vocabulaire :", vocab_size)

Exemple word2idx : [('<OOV>', 1), ('movie', 2), ('film', 3), ('one', 4), ('like', 5)]
Exemple idx2word : [(1, '<OOV>'), (2, 'movie'), (3, 'film'), (4, 'one'), (5, 'like')]
Taille du vocabulaire : 10000


The tokenizer's **texts_to_sequences** method allows each review to be transformed into a sequence of integers (word indices). Only words known to the tokenizer will be considered.

<span style="color:gray">tokenizer.texts_to_sequences(['hello my dear readers'])<br>
<span style="color:gray">==> [[4422, 2974, 6117]]

Here, the index of the words "hello", "dear" and "readers" are 4422, 2974 and 6117. Since the word "my" is a stopword, it is not in the dictionary and will therefore be ignored.

* Transform each review in **X_text_train** into a sequence of integers using the **texts_to_sequences method**. Store the result in the variable **X_train**.<br>
* Do the same for **X_text_test** and store the list in **X_test**.

In [31]:
X_train = tokenizer.texts_to_sequences(X_text_train)
X_test = tokenizer.texts_to_sequences(X_text_test)


The **pad_sequences** function from **tensorflow.keras.preprocessing.sequence** transforms a list of sequences into a Numpy array with shapes (num_samples, num_timesteps).<br> **num_timesteps** is either the **maxlen** argument if provided, or the length of the longest sequence otherwise.

**Exemple:**<br>
<span style="color:gray">sequence = [[4,8,1,2], [15], [8, 5, 6]]<br>
<span style="color:gray">tf.keras.preprocessing.sequence.pad_sequences(sequence, maxlen=3, padding='post', truncating='post')<br>
<span style="color:gray">==> ([[4, 8, 1],
           [15, 0, 0],
           [8, 5, 6]])

Sequences shorter than **maxlen** are padded with a default value of 0, while sequences longer than **maxlen** are truncated to match the desired length. More information here.https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences

pad_sequences(

              sequences,        # List of texts converted into indices
             
              maxlen=None,      # Maximum number of words in a text.
             
              dtype='int32',    # returns an output of dtype.
              
              padding='pre',    # Pad sequences at the beginning ('pre') or at the end ('post').
              
              truncating='pre', # truncate at the beginning, truncate at the end if the sequence is too long.
              
              value=0.0         # Pad sequences with value.
)

* Transform the sequence list **X_train** into a NumPy array using the **pad_sequences** function. Specify in the function arguments **maxlen=500**, **padding='post'**, and **truncating='post'**.<br>
* Store the result in the variable **X_train**. <br>
* Do the same for **X_test**.

In [32]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(
    X_train,
    maxlen=500,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=500,
    padding='post',
    truncating='post'
)

print("Shape de X_train :", X_train.shape)
print("Shape de X_test :", X_test.shape)


Shape de X_train : (10000, 500)
Shape de X_test : (40000, 500)


### Modele : RNN
The model will be built as follows:

![image.png](attachment:03ce25f2-b9e6-4c39-9614-a21c639d91ad.png)

1. The **Embedding** layer will transform each word index into an embedding vector. The embedding matrix **W** will be learned as the model trains. The resulting dimensions are: **(batch, sequence, embedding)**.<br>
2. Then, the **RNN** layer will go through the sequence of encoded words to capture the relationships between different words. The **RNN** layer will return a new embedding for each word. The resulting dimensions will be: **(batch, sequence, rnn_embedding)**.<br>
3. Next, the **GlobalAveragePooling1D** layer returns a fixed-length output vector for each example by averaging over the sequence dimension. This transformation acts as a **bag-of-words** and allows the model to handle variable-length inputs.<br>
4. Finally, since we are dealing with a classification problem (positive or negative review), we will add **Dense** layers to classify the sentiment of the review.

* Define a **Sequential** model named **model**.<br>
* Add an **Embedding** layer specifying an input size of 10,000 and an output size of 200.<br>
* Add an **RNN **layer with a **GRUCell** of 128 neurons.The **RNN** layer should return a sequence (return_sequences argument).<br>
* Add a **Dropout** layer to prevent overfitting.<br>
* Add a **GlobalAveragePooling1D** layer to apply our **bag of words**.<br>
* Add a **Dense** layer with 256 neurons and a **relu** activation function.<br>
* Add a **Dropout** layer to prevent overfitting.<br>
* Add a final **Dense** layer with a **softmax** activation function.<br>
* Display the model summary.<br>

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, RNN, GRUCell, Dropout, GlobalAveragePooling1D, Dense

model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=200))

model.add(RNN(GRUCell(128), return_sequences=True))

model.add(Dropout(0.5))

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation='relu'))

model.add(Dropout(0.5))

model.add(Dense(2, activation='softmax'))

model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_5 (RNN)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

* From **sklearn.preprocessing** import **LabelEncoder** to convert y to integer.
* Compile the model with an **'adam'** optimizer, a **'sparse_categorical_crossentropy'** loss function, and an **'accuracy'** metric.<br>
* Train the model for a few epochs.

In [34]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model.pop()
model.add(Dense(2, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_enc,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

loss, acc = model.evaluate(X_test, y_test_enc)
print("Test accuracy :", acc)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 81ms/step - accuracy: 0.4909 - loss: 0.6980 - val_accuracy: 0.5205 - val_loss: 0.6902
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.5579 - loss: 0.6729 - val_accuracy: 0.7605 - val_loss: 0.5310
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.8553 - loss: 0.3703 - val_accuracy: 0.8735 - val_loss: 0.3211
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.9409 - loss: 0.1763 - val_accuracy: 0.8670 - val_loss: 0.4010
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.9772 - loss: 0.0833 - val_accuracy: 0.8570 - val_loss: 0.4708
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.8555 - loss: 0.4802
Test accuracy : 0.8605250120162964


* Evaluate the model on the validation sample.

In [35]:
loss, acc = model.evaluate(X_test, y_test_enc)
print("Test accuracy :", acc)

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.8555 - loss: 0.4802
Test accuracy : 0.8605250120162964


Le modèle atteint une précision d’environ 86 %, montrant qu’il généralise bien sur les données de test. La perte de 0,48 indique qu’il y a encore une marge de progression pour réduire les erreurs de prédiction.

* Now test the model by some data to detect if the sentiment is positive or negative. Example:<br>
new_sentences = [
    
    "I loved this movie! It was amazing.",
    
    "The product is terrible and broke immediately.",
    
    "Not bad, could be better."
]


In [36]:
new_sentences = [
    "I loved this movie! It was amazing.",
    "The product is terrible and broke immediately.",
    "Not bad, could be better."
]

new_sentences_clean = [preprocess_sentence(s) for s in new_sentences]

new_sequences = tokenizer.texts_to_sequences(new_sentences_clean)

new_padded = pad_sequences(
    new_sequences,
    maxlen=500,
    padding='post',
    truncating='post'
)

predictions = model.predict(new_padded)

predicted_classes = np.argmax(predictions, axis=1)

predicted_labels = le.inverse_transform(predicted_classes)

for sentence, label in zip(new_sentences, predicted_labels):
    print(f"Sentence: {sentence}")
    print(f"Predicted sentiment: {label}")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 715ms/step
Sentence: I loved this movie! It was amazing.
Predicted sentiment: positive
--------------------------------------------------
Sentence: The product is terrible and broke immediately.
Predicted sentiment: negative
--------------------------------------------------
Sentence: Not bad, could be better.
Predicted sentiment: negative
--------------------------------------------------
